In [1]:
import numpy as np
import os
import subprocess
import pandas as pd
import random
import glob

# =====================================================
# 1️⃣ تعريف الشبكة (0 = ممر، 1 = جدار، 2 = محل، 3 = فتحة)
# =====================================================
grid = np.array([
    [1,1,1,3,1,3,1,3,1,3,1,3,1,3,1,1,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1,1],
    [1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1],
    [1,1,1,0,2,0,2,0,2,0,2,0,2,0,2,0,1,1,1],
    [1,1,1,3,1,3,1,1,1,3,1,3,1,3,1,1,1,1,1]
])
rows, cols = grid.shape

# =====================================================
# 2️⃣ إنشاء مجلد المحاكاة
# =====================================================
os.makedirs("fds_cases", exist_ok=True)

# =====================================================
# 3️⃣ اختيار محل عشوائي للحريق
# =====================================================
shops = [(i,j) for i in range(rows) for j in range(cols) if grid[i,j]==2]
fire_shop = random.choice(shops)
i_fire, j_fire = fire_shop
print("Fire location:", fire_shop)

# =====================================================
# 4️⃣ إنشاء ملف FDS
# =====================================================
fds_file = "fds_cases/fire_model.fds"

with open(fds_file,"w") as f:
    f.write("&HEAD CHID='fire_model', TITLE='Mall Fire Simulation'/\n")
    f.write(f"&MESH IJK={cols},{rows},5 XB=0,{cols},0,{rows},0,3 /\n")
    f.write("&TIME T_END=60 /\n")  # مدة 60 ثانية لتسريع الانتشار
    f.write("&REAC FUEL='PROPANE' /\n")
    f.write("&SURF ID='FIRE', HRRPUA=500 /\n")

    # إنشاء الجدران والفتحات
    for r in range(rows):
        for c in range(cols):
            y = rows - r
            if grid[r,c] == 1:
                f.write(f"&OBST XB={c},{c+1},{y-1},{y},0,3 /\n")
            if grid[r,c] == 3:
                f.write(f"&OBST XB={c},{c+1},{y-1},{y},0,3 SURF_ID='OPEN'/\n")

    # إضافة الحريق في المحل
    x_offset = random.uniform(0,0.8)
    y_offset = random.uniform(0,0.8)
    f.write(f"&OBST XB={j_fire+x_offset},{j_fire+0.2+x_offset},{rows-i_fire-1+y_offset},{rows-i_fire-0.2+y_offset},0,0.5 SURF_ID='FIRE'/\n")

    # إضافة DEVC لكل الممرات (0)
    for r in range(rows):
        for c in range(cols):
            if grid[r,c] == 0:
                x = c + 0.5
                y = rows - r - 0.5
                f.write(f"&DEVC ID='Cell_{r}_{c}', QUANTITY='TEMPERATURE', XYZ={x},{y},0.5 /\n")

    f.write("&TAIL /\n")

print("FDS file created")

# =====================================================
# 5️⃣ تشغيل المحاكاة
# =====================================================
print("Running FDS...")
subprocess.run([r"FDS6\bin\fds_local.bat", fds_file], check=True)
print("Simulation finished")

# =====================================================
# 6️⃣ البحث عن ملف CSV الناتج
# =====================================================
csv_files = glob.glob("fds_cases/*_devc.csv")
if len(csv_files) == 0:
    print("No DEVC CSV file found")
    exit()
csv_file = csv_files[0]
print("CSV found:", csv_file)

# =====================================================
# 7️⃣ قراءة CSV وتحويله إلى مصفوفة خطر
# =====================================================
df = pd.read_csv(csv_file)

# إزالة أي علامات اقتباس حول أسماء الأعمدة
df.columns = df.columns.str.replace('"', '').str.strip()

time_steps = df.shape[0]
risk_grid = np.zeros((time_steps, rows, cols))

for r in range(rows):
    for c in range(cols):
        matches = [col for col in df.columns if col.startswith(f"Cell_{r}_{c}")]
        if matches:
            risk_grid[:,r,c] = df[matches[0]].values

# =====================================================
# 8️⃣ إنشاء Dataset للتدريب
# =====================================================
training_data = []

for t in range(time_steps):
    for r in range(rows):
        for c in range(cols):
            if grid[r,c] == 0:
                training_data.append({
                    "fire_row": i_fire,
                    "fire_col": j_fire,
                    "time": t,
                    "cell_row": r,
                    "cell_col": c,
                    "risk": risk_grid[t,r,c]
                })

training_df = pd.DataFrame(training_data)
training_df.to_csv("fire_training_dataset.csv", index=False)
print("Dataset saved successfully")

Fire location: (6, 12)
FDS file created
Running FDS...
Simulation finished
No DEVC CSV file found


IndexError: list index out of range